# 03 - ELO Prediction Model

Fit a regression model that predicts a player's ELO from their Average Centipawn Loss (ACPL).

Uses the 200k sampled Lichess games with real Stockfish evals to calibrate:
1. Compute ACPL per player per game from the moves CSV (real Stockfish centipawns)
2. Join with known ELO from the games CSV
3. Fit `ELO = a - b × ln(ACPL)` using nonlinear regression
4. Also fit a multivariate model (ACPL + blunder rate + captures + checks)
5. PCA player profile clustering
6. Export coefficients to `elo_model.json` for the React frontend

## Setup

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("scikit-learn")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import json
import os

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print('Setup done.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Learning-document/Data Visualization/Project 2/data'
FRONTEND_DIR = os.path.join(DATA_DIR, 'frontend')
os.makedirs(FRONTEND_DIR, exist_ok=True)

games = pd.read_csv(os.path.join(DATA_DIR, 'lichess_sampled_games.csv'))
moves = pd.read_csv(os.path.join(DATA_DIR, 'lichess_sampled_moves.csv'))

print(f'Games: {len(games):,}')
print(f'Moves (eval): {len(moves):,}')
print(f'\nMoves columns: {list(moves.columns)}')
print(f'Games columns: {list(games.columns)}')

## 1. Compute ACPL per player per game

From the moves CSV:
- `eval_change_for_player`: centipawn change from the player's perspective (negative = blunder)
- ACPL = average of max(0, -eval_change) across all moves in the game
- Join with the player's known ELO from the games CSV

In [ ]:
# Cap individual eval changes at ±1000cp to reduce noise from mate scores
MAX_EVAL_CHANGE = 1000
MIN_MOVES = 10  # skip very short games

moves['eval_change_clipped'] = moves['eval_change_for_player'].clip(-MAX_EVAL_CHANGE, MAX_EVAL_CHANGE)
moves['loss_cp'] = moves['eval_change_clipped'].clip(lower=0) * -1  # positive = centipawns lost
moves['loss_cp'] = moves['eval_change_clipped'].apply(lambda x: max(0, -x))

# Group by (game_idx, is_white) = one player's performance in one game
player_stats = moves.groupby(['game_idx', 'is_white']).agg(
    acpl=('loss_cp', 'mean'),
    n_moves=('loss_cp', 'count'),
    total_loss=('loss_cp', 'sum'),
    blunder_count=('eval_change_clipped', lambda x: (x < -200).sum()),
    capture_count=('is_capture', 'sum'),
    check_count=('is_check', 'sum'),
).reset_index()

player_stats['blunder_rate'] = player_stats['blunder_count'] / player_stats['n_moves']
player_stats['capture_pct'] = player_stats['capture_count'] / player_stats['n_moves']
player_stats['check_pct'] = player_stats['check_count'] / player_stats['n_moves']

# Filter short games
player_stats = player_stats[player_stats['n_moves'] >= MIN_MOVES]

# Join with ELO
def get_elo(row):
    game = games.iloc[int(row['game_idx'])]
    return game['white_rating'] if row['is_white'] else game['black_rating']

player_stats['elo'] = player_stats.apply(get_elo, axis=1)

# Filter unreasonable values
mask = (player_stats['acpl'] > 0) & (player_stats['acpl'] < 1000) & \
       (player_stats['elo'] > 400) & (player_stats['elo'] < 3200)
df = player_stats[mask].copy()

print(f'Player-game records: {len(df):,}')
print(f'\nACPL stats:')
print(df['acpl'].describe().to_string())
print(f'\nELO stats:')
print(df['elo'].describe().to_string())